# Faruq-v3 ACMC1 -- one-stage screening seed 42

Melatih satu kandidat yang field-level dan end-to-end: fusi P3/P4/P5 hanya untuk koreksi klasifikasi berbasis ambiguitas. Tidak memakai ROI/crop, top-K kandidat, atau decode box sebelum klasifikasi. Regresi box asli YOLO26 dipertahankan. Test tidak tersedia dan tidak dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_summary.json',
    'experiments/faruq-v3-acmc-one-stage-v1/static_audit.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
BASELINE = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/val_reports/D0_seed42_summary.json')
STATIC_AUDIT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-one-stage-v1/static_audit.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-one-stage-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('D0        :', D0_CHECKPOINT)
print('STATIC    :', STATIC_AUDIT)
print('OUTPUT    :', OUTPUT_ROOT)
last = OUTPUT_ROOT / 'ACMC1_seed42/weights/last.pt'
best = OUTPUT_ROOT / 'ACMC1_seed42/weights/best.pt'
print('ACMC1     :', 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START'))

In [ ]:
import json
static = json.loads(STATIC_AUDIT.read_text(encoding='utf-8'))
assert static['training_executed'] is False
assert static['dataset_accessed'] is False
assert static['test_images_accessed'] is False
assert static['decision'] == 'PASS', 'STOP: static audit ACMC belum PASS.'
print('STATIC GATES:', static['gates'])
print('PARAMETERS  :', static['parameter_counts'])
print('PASS: dataset development boleh dipakai untuk seed 42.')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_acmc',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--baseline-summary', str(BASELINE),
    '--d0-checkpoint', str(D0_CHECKPOINT),
    '--static-audit', str(STATIC_AUDIT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout, end='', flush=True)
if process.returncode != 0:
    raise RuntimeError(f'ACMC1 gagal dengan return code {process.returncode}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/acmc1_seed42_decision.json'
assert SUMMARY.is_file(), f'ACMC1 belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = [{'model': name, **metrics} for name, metrics in result['results'].items()]
metrics = ('macro_map50_95', 'bottom3_class_map50_95', 'worst_class_map50_95')
display(pd.DataFrame(rows).style.format({name: '{:.2%}' for name in metrics}))
print('DELTAS  :', result['deltas'])
print('CRITERIA:', result['criteria'])
print('DECISION:', result['decision'])
print('NEXT    :', result['next_action'])
print('SUMMARY :', SUMMARY)
print('Kirim tabel dan keputusan. Jangan membuka test atau menjalankan seed lain.')